# Information Gain & Entropy Calculation

A from-scratch implementation of entropy and information gain — the metrics used to decide which feature to split on first when building a decision tree.

---

## What is information gain?

Information gain measures how much a particular feature helps predict the target variable. The higher the information gain, the more that feature reduces uncertainty.

## Why calculate it?

It's used to decide which feature should sit at the top of a decision tree — the feature with the greatest information gain reduces uncertainty the most, so it goes first.

## How it's calculated

IG(S,A) = H(S) - weighted average of H(Sv)

where:
- H(S) = entropy of the original dataset
- H(Sv) = entropy of each subset based on its proportion of samples
- weighted average = proportion of samples in each subset

---

**Entropy:** entropy(X) = &minus; &sum; p_i log2(p_i)

**Information Gain:** IG(X, a) = entropy(X) &minus; &sum; (|X_v| / |X|) &middot; entropy(X_v)


## Step 1: Import Libraries

In [ ]:
from math import log2

## Step 2: Create the Dataset

Creating lists for each feature as given in the assignment.

In [ ]:
# Creating the loan dataset with 10 applicants
Employment_Status = ['Employed', 'Unemployed', 'Employed', 'Unemployed', 'Employed',
                     'Unemployed', 'Employed', 'Employed', 'Unemployed', 'Employed']

Housing_Status = ['Owner', 'Rented', 'Rented', 'Owner', 'Owner',
                  'Rented', 'Rented', 'Owner', 'Owner', 'Rented']

Bank_Account = ['Active', 'None', 'Active', 'None', 'None',
                'None', 'Active', 'Active', 'None', 'Active']

Loan_Default = ['Repaid', 'Defaulted', 'Repaid', 'Defaulted', 'Repaid',
                'Defaulted', 'Repaid', 'Repaid', 'Defaulted', 'Repaid']

print('Dataset created with',len(Loan_Default),'samples')

Dataset created with 10 samples


## Step 3: Calculate Entropy

**What does entropy represent?**

Entropy is a way of measuring the amount of randomness in the data.

**Why is entropy important?**

Knowing the entropy of the data before and after splitting will allow us to compute the "information gain", which allows us to determine if we have made the right decision when making our tree.

**How can I find entropy?**

H(S)= -p(Repaid)*log2(p(Repaid))-p(Defaulted)*log2(p(Defaulted)).

**What does the entropy equation mean?**

If all the samples in a dataset belong to one class, then entropy = 0 (the data is pure).

If the data set contains equal amounts of two classes, then entropy = 1 (the data is maximally impure).

The use of log base 2 (log2) to compute the entropy is based on the fact that we quantify the amount of information as "bits"

In [ ]:
# Function to calculate entropy
def calculate_entropy(target_column):
    total = len(target_column)

    # Count how many Repaid and Defaulted
    repaid_count = 0
    defaulted_count = 0

    for value in target_column:
        if value == 'Repaid':
            repaid_count = repaid_count + 1
        else:
            defaulted_count = defaulted_count + 1

    # Calculate probabilities
    p_repaid = repaid_count / total
    p_defaulted = defaulted_count / total

    # Calculate entropy using formula
    entropy = 0
    if p_repaid > 0:
        entropy = entropy - p_repaid * log2(p_repaid)
    if p_defaulted > 0:
        entropy = entropy - p_defaulted * log2(p_defaulted)

    return entropy

print('Entropy function created')

Entropy function created


## Step 4: Calculate Parent Entropy

This is the entropy of the whole dataset before any split.

In [ ]:
parent_entropy = calculate_entropy(Loan_Default)
print('Parent Entropy =',parent_entropy)

Parent Entropy = 0.9709505944546686


## Step 5: Determine Information Gain for Each Attribute

**What is happening here?**

Each attribute will have the information gain calculated by determining how much the information gain from the parent entropy has been reduced.

**Why do it this way?**

To determine which attributes are most relevant, you first need to see if there is a unique value of an attribute that impacts the target variable, and then you can determine the entropy for each subset of the target variable based on the unique value of that attribute.

**The process to accomplish this:**

1. Determine all the unique values of the attribute.

2. Create subsets of the target variable for each unique value of the attribute.

3. Determine the entropy of each subset of the target variable.

4. Take the weighted average of the entropies of each subset (the weight is the number of samples in each subset).

5. The information gain is equal to the parent entropy minus the weighted average.

In [ ]:
# Function to calculate information gain
def calculate_information_gain(feature, target):
    total = len(target)
    parent_entropy = calculate_entropy(target)

    # Find unique values in feature
    unique_values = []
    for value in feature:
        if value not in unique_values:
            unique_values.append(value)

    # Calculate weighted entropy for each value
    weighted_entropy = 0

    for value in unique_values:
        # Create subset for this value
        subset = []
        for i in range(len(feature)):
            if feature[i] == value:
                subset.append(target[i])

        # Calculate entropy of this subset
        subset_entropy = calculate_entropy(subset)

        # Calculate weight
        weight = len(subset) / total

        # Add to weighted entropy
        weighted_entropy = weighted_entropy + weight * subset_entropy

        print(value,': samples=',len(subset),', entropy=',round(subset_entropy,4))

    # Calculate information gain
    information_gain = parent_entropy - weighted_entropy

    return information_gain

print('Information gain function created')

Information gain function created


### Feature 1: Employment Status

**Logic for choosing this feature:**  
Employment status might be important because people with jobs are more likely to repay loans.

In [ ]:
print('\nEmployment_Status:')
ig_employment = calculate_information_gain(Employment_Status, Loan_Default)
print('Information Gain =',round(ig_employment,6))


Employment_Status:
Employed : samples= 6 , entropy= 0.0
Unemployed : samples= 4 , entropy= 0.0
Information Gain = 0.970951


### Feature 2: Housing Status

**Logic for choosing this feature:**  
People who own houses might be more financially stable and repay loans.

In [ ]:
print('\nHousing_Status:')
ig_housing = calculate_information_gain(Housing_Status, Loan_Default)
print('Information Gain =',round(ig_housing,6))


Housing_Status:
Owner : samples= 5 , entropy= 0.971
Rented : samples= 5 , entropy= 0.971
Information Gain = 0.0


### Feature 3: Bank Account

**Logic for choosing this feature:**  
Having an active bank account shows financial activity and might affect loan repayment.

In [ ]:
print('\nBank_Account:')
ig_bank = calculate_information_gain(Bank_Account, Loan_Default)
print('Information Gain =',round(ig_bank,6))


Bank_Account:
Active : samples= 5 , entropy= 0.0
None : samples= 5 , entropy= 0.7219
Information Gain = 0.609987


## Step 6: Compare & Pick the "Best" Feature

## What Are We Doing?

We are comparing each value for our three features (IG), to see which has the highest number.

## Why This Method?

Each feature that produces a high information gain will reduce our uncertainty by the most amount, therefore it makes sense to use these as the starting point for our Decision Tree, i.e., the Root Node.

## How Do I Interpret My Results?:

- The higher the IG the better the feature.

- The closer to 0 the IG is the less helpful the feature is.

- The closer to 1 the IG is the more helpful the feature is.

In [ ]:
print('\n==========================================')
print('RESULTS SUMMARY')
print('==========================================')
print('Employment_Status  :',round(ig_employment,6))
print('Housing_Status     :',round(ig_housing,6))
print('Bank_Account       :',round(ig_bank,6))

# Find which one is highest
if ig_employment > ig_housing and ig_employment > ig_bank:
    best_feature = 'Employment_Status'
    best_ig = ig_employment
elif ig_housing > ig_employment and ig_housing > ig_bank:
    best_feature = 'Housing_Status'
    best_ig = ig_housing
else:
    best_feature = 'Bank_Account'
    best_ig = ig_bank

print('\nBest Feature =',best_feature)
print('Information Gain =',round(best_ig,6))
print('==========================================')


RESULTS SUMMARY
Employment_Status  : 0.970951
Housing_Status     : 0.0
Bank_Account       : 0.609987

Best Feature = Employment_Status
Information Gain = 0.970951


## Conclusion

**What did we find?**: The feature that has the largest information gain (IG) will serve as the root node of the decision tree.

**Why is this important?**: Beginning with the most informative feature reduces the amount of uncertainty the first time the data set is divided into two groups (or branches), and thus leads to an increase in accuracy for the decision tree.

**How would we use this?**:

1. Determine which feature is the best one (has the greatest IG).

2. Create a branch for each possible value of the best feature.

3. Repeat the above steps for all of the other features in each branch.

4. Continue until there are no additional features to evaluate, or all cases in a given branch have been assigned to a single category.